In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import datetime

def fetch_forex_factory_data(start_date, end_date):
    url = f'https://www.forexfactory.com/calendar.php?month={start_date.month}.{start_date.year}&week={start_date.isocalendar()[1]}'
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    events = []
    for row in soup.find_all('tr', class_='calendar_row'):
        event = {}
        try:
            event['Date'] = row.find('td', class_='calendar__date').get_text(strip=True)
            event['time'] = row.find('td', class_='calendar__time').get_text(strip=True)
            event['currency'] = row.find('td', class_='calendar__currency').get_text(strip=True)
            event['impact'] = row.find('td', class_='impact').get_text(strip=True)
            event['event'] = row.find('td', class_='calendar__event').get_text(strip=True)
            event['actual'] = row.find('td', class_='calendar__actual').get_text(strip=True).replace('N/A', 'nan')
            event['forecast'] = row.find('td', class_='calendar__forecast').get_text(strip=True).replace('N/A', 'nan')
            event['previous'] = row.find('td', class_='calendar__previous').get_text(strip=True).replace('N/A', 'nan')
            events.append(event)
        except AttributeError:
            continue

    return events

In [2]:
# Fetch data for the last month
end_date = datetime.datetime.now()
start_date = end_date - datetime.timedelta(days=5)

forex_data = fetch_forex_factory_data(start_date, end_date)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
# Convert to DataFrame and filter for the date range
forex_df = pd.DataFrame(forex_data)
forex_df['datetime'] = pd.to_datetime(forex_df['Date'] + ' ' + forex_df['time'])
forex_df = forex_df[(forex_df['datetime'] >= start_date) & (forex_df['datetime'] <= end_date)]

In [ ]:
# Clean and convert columns to numeric
forex_df['actual'] = pd.to_numeric(forex_df['actual'], errors='coerce')
forex_df['forecast'] = pd.to_numeric(forex_df['forecast'], errors='coerce')
forex_df['previous'] = pd.to_numeric(forex_df['previous'], errors='coerce')

In [3]:
# Calculate the differences
forex_df['actual_previous_diff'] = forex_df['actual'] - forex_df['previous']
forex_df['actual_forecast_diff'] = forex_df['actual'] - forex_df['forecast']

forex_df.to_csv('forex_factory_data.csv', index=False)

KeyError: 'Date'

In [ ]:
# Load the Forex Factory data
forex_df = pd.read_csv('forex_factory_data.csv')

# Clean and convert columns to numeric
forex_df['actual'] = pd.to_numeric(forex_df['actual'], errors='coerce')
forex_df['forecast'] = pd.to_numeric(forex_df['forecast'], errors='coerce')
forex_df['previous'] = pd.to_numeric(forex_df['previous'], errors='coerce')

# Calculate the differences
forex_df['actual_previous_diff'] = forex_df['actual'] - forex_df['previous']
forex_df['actual_forecast_diff'] = forex_df['actual'] - forex_df['forecast']

forex_df.head()


In [ ]:
# Load the multipliers dataset
multipliers_df = pd.read_csv('E:\Economic_Data\Output data\modelling output\pca_ridge_regression_results.csv')
multipliers_df['Coefficients'] = multipliers_df['Coefficients'].str.replace(r'(\d)\s+(\d)', r'\1, \2', regex=True)
multipliers_df['Coefficients'] = multipliers_df['Coefficients'].apply(eval)

# Function to calculate the expected percentage change
def calculate_expected_change(row, actual_prev, actual_forecast):
    coefficients = row['Coefficients']
    intercept = row['Intercept']
    if len(coefficients) == 2:
        coef_actual_prev, coef_actual_forecast = coefficients
        expected_change = (coef_actual_prev * actual_prev + coef_actual_forecast * actual_forecast + intercept)
    else:
        coef_actual_prev = coefficients[0]
        coef_actual_forecast = 0  # Set the other coefficient to zero
        expected_change = (coef_actual_prev * actual_prev + coef_actual_forecast * actual_forecast + intercept)
    return expected_change

# Merge Forex Factory data with multipliers
merged_df = pd.merge(forex_df, multipliers_df, left_on='event', right_on='Event')

# Apply the function to calculate expected changes
merged_df['Expected_Change'] = merged_df.apply(
    lambda row: calculate_expected_change(row, row['actual_previous_diff'], row['actual_forecast_diff']), axis=1
)

merged_df.head()


In [ ]:
# Load the Bitcoin price data
btc_price_data = pd.read_csv('E:\SignalModel\price 2024-05-01, 2024-06-01 min.csv')
btc_price_data['datetime'] = pd.to_datetime(btc_price_data['datetime'])
btc_price_data.set_index('datetime', inplace=True)

# Function to calculate the percentage change in BTC price
def calculate_percentage_change(df, interval):
    df[f'change_{interval}m'] = df['close'].pct_change(periods=interval) * 100
    return df

# Calculate the actual percentage changes for different intervals
for interval in [5, 15, 30, 60]:
    btc_price_data = calculate_percentage_change(btc_price_data, interval)

# Function to align BTC price data with economic events data
def align_btc_price_with_events(btc_data, events_data, interval):
    results = []
    for _, event in events_data.iterrows():
        event_time = pd.Timestamp(event['date'] + ' ' + event['time'])
        interval_end_time = event_time + pd.Timedelta(minutes=interval)
        actual_change = btc_data.loc[interval_end_time, f'change_{interval}m'] if interval_end_time in btc_data.index else None
        expected_change = event['Expected_Change']
        results.append((event_time, interval, actual_change, expected_change))
    return pd.DataFrame(results, columns=['Event_Time', 'Interval', 'Actual_Change', 'Expected_Change'])

# Align BTC price data with economic events data for each interval
backtest_results = pd.DataFrame()
for interval in [5, 15, 30, 60]:
    interval_results = align_btc_price_with_events(btc_price_data, merged_df, interval)
    backtest_results = pd.concat([backtest_results, interval_results])

# Save the backtest results to a CSV file
backtest_results.to_csv('backtest_results.csv', index=False)
